# Good Food, Good Gut - Competition Notebook

This notebook builds a classification pipeline to predict inspection outcome (`target`):
- `1` = pass
- `0` = fail

It also answers all required analysis questions and exports a Kaggle-ready `submission.csv`.

Primary metric: **F1 score**.

In [3]:
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction import DictVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
sample = pd.read_csv('sample_submission.csv')

print('Train shape:', train.shape)
print('Test shape:', test.shape)
print('Target distribution:')
print(train['target'].value_counts(normalize=True).rename('ratio'))

ModuleNotFoundError: No module named 'numpy'

In [ ]:
def parse_violation_list(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return [str(v) for v in value if str(v).strip()]

    text = str(value).strip()
    if text in {'', '[]', 'nan', 'None'}:
        return []

    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list):
            return [str(v).strip() for v in parsed if str(v).strip()]
    except Exception:
        pass

    cleaned = text.strip('[]')
    if not cleaned:
        return []
    parts = [p.strip().strip("'").strip('\"') for p in cleaned.split(',')]
    return [p for p in parts if p]


def canonical_code(code):
    s = str(code).strip()
    if not s:
        return s
    if s.isdigit():
        return str(int(s))
    return s


def add_features(df):
    out = df.copy()

    out['viol_codes'] = out['Violation_List'].apply(parse_violation_list)
    out['viol_codes'] = out['viol_codes'].apply(lambda arr: [canonical_code(x) for x in arr if canonical_code(x) != ''])
    out['viol_count'] = out['viol_codes'].apply(len)
    out['unique_viol_count'] = out['viol_codes'].apply(lambda x: len(set(x)))
    out['has_any_violation'] = (out['viol_count'] > 0).astype(int)

    out['City'] = out['City'].fillna('UNKNOWN').astype(str).str.upper().str.strip()
    out['Facility Type'] = out['Facility Type'].fillna('UNKNOWN').astype(str).str.strip()
    out['Risk'] = out['Risk'].fillna('UNKNOWN').astype(str).str.strip()
    out['Inspection Type'] = out['Inspection Type'].fillna('UNKNOWN').astype(str).str.strip()
    out['State'] = out['State'].fillna('UNKNOWN').astype(str).str.strip()

    for c in ['Zip', 'Latitude', 'Longitude', 'year', 'month', 'weekday']:
        out[c] = pd.to_numeric(out[c], errors='coerce')

    return out


train_df = add_features(train)
test_df = add_features(test)

train_df[['viol_count', 'unique_viol_count']].describe().T

## Required Analytical Questions (1 to 5)

In [ ]:
failed = train_df[train_df['target'] == 0].copy()

# 1) Top 10 violation codes associated with failed inspections
q1_top10 = failed['viol_codes'].explode().dropna().value_counts().head(10)
print('Q1 - Top 10 codes among failed inspections')
display(q1_top10.to_frame('failed_count'))

# 2) Facility Type with highest number of unique violation codes
q2 = (train_df.explode('viol_codes')
      .dropna(subset=['viol_codes'])
      .groupby('Facility Type')['viol_codes']
      .nunique()
      .sort_values(ascending=False))
print('Q2 - Facility type with highest unique violation codes:')
display(q2.head(10).to_frame('unique_violation_codes'))

# 3) Risk category with highest failure rate (filtered by at least 30 records)
risk_stats = (train_df.groupby('Risk')['target']
              .agg(count='size', fail_rate=lambda s: (s == 0).mean())
              .sort_values(['fail_rate', 'count'], ascending=[False, False]))
q3 = risk_stats[risk_stats['count'] >= 30]
print('Q3 - Risk category failure rate (count >= 30)')
display(q3.head(10))

# 4) Months and weekdays with most failures
month_stats = train_df.groupby('month')['target'].agg(count='size', fail_rate=lambda s: (s == 0).mean())
weekday_stats = train_df.groupby('weekday')['target'].agg(count='size', fail_rate=lambda s: (s == 0).mean())

print('Q4 - Month failure rates')
display(month_stats.sort_values('fail_rate', ascending=False))
print('Q4 - Weekday failure rates')
display(weekday_stats.sort_values('fail_rate', ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
month_stats['fail_rate'].plot(kind='bar', ax=axes[0], color='#D95F02', title='Failure Rate by Month')
weekday_stats['fail_rate'].plot(kind='bar', ax=axes[1], color='#1B9E77', title='Failure Rate by Weekday')
axes[0].set_ylabel('fail_rate')
axes[1].set_ylabel('fail_rate')
plt.tight_layout()
plt.show()

# 5) Outcomes across city / ZIP regions (count >= 30 for stability)
city_stats = (train_df.groupby('City')['target']
              .agg(count='size', fail_rate=lambda s: (s == 0).mean())
              .query('count >= 30')
              .sort_values(['fail_rate', 'count'], ascending=[False, False]))

zip_stats = (train_df.groupby('Zip')['target']
             .agg(count='size', fail_rate=lambda s: (s == 0).mean())
             .query('count >= 30')
             .sort_values(['fail_rate', 'count'], ascending=[False, False]))

print('Q5 - Cities with highest failure rates')
display(city_stats.head(10))
print('Q5 - ZIPs with highest failure rates')
display(zip_stats.head(10))

## Required Analytical Questions (6 to 10)

In [ ]:
# 6) Are more violations linked to failure?
viol_fail = (train_df.groupby('viol_count')['target']
            .agg(count='size', fail_rate=lambda s: (s == 0).mean())
            .reset_index())
corr_vc_fail = np.corrcoef(train_df['viol_count'], (train_df['target'] == 0).astype(int))[0, 1]
print('Q6 - Correlation (viol_count vs fail):', round(corr_vc_fail, 4))
display(viol_fail.head(15))

plt.figure(figsize=(8, 4))
sns.lineplot(data=viol_fail, x='viol_count', y='fail_rate', marker='o')
plt.title('Failure Rate vs Number of Violations')
plt.ylim(0, 1)
plt.show()

# 7) Most common violations among different facility types
facility_code_top = (train_df.explode('viol_codes')
                     .dropna(subset=['viol_codes'])
                     .groupby(['Facility Type', 'viol_codes'])
                     .size()
                     .reset_index(name='count')
                     .sort_values(['Facility Type', 'count'], ascending=[True, False]))

print('Q7 - Top 3 violations per facility type (preview)')
display(facility_code_top.groupby('Facility Type').head(3).head(30))

# 8) Geographic trends with latitude and longitude
geo_cmp = train_df.groupby('target')[['Latitude', 'Longitude']].mean()
print('Q8 - Mean coordinates by outcome (target=0 fail, 1 pass)')
display(geo_cmp)

plt.figure(figsize=(7, 5))
sample_geo = train_df.sample(min(15000, len(train_df)), random_state=42)
sns.scatterplot(
    data=sample_geo, x='Longitude', y='Latitude', hue='target',
    alpha=0.35, s=15, palette={0: '#E41A1C', 1: '#377EB8'}
)
plt.title('Q8 - Geographic distribution of inspection outcomes (sampled)')
plt.legend(title='target')
plt.show()

# 9) Inspection type vs outcome
insp_stats = (train_df.groupby('Inspection Type')['target']
              .agg(count='size', fail_rate=lambda s: (s == 0).mean())
              .query('count >= 30')
              .sort_values(['fail_rate', 'count'], ascending=[False, False]))
print('Q9 - Inspection types with highest failure rates (count >= 30)')
display(insp_stats.head(15))

# 10) Strongest violation code indicators of failure
base_fail = (train_df['target'] == 0).mean()
code_fail_stats = (train_df.explode('viol_codes')
                   .dropna(subset=['viol_codes'])
                   .groupby('viol_codes')['target']
                   .agg(count='size', fail_rate=lambda s: (s == 0).mean())
                   .assign(fail_uplift=lambda d: d['fail_rate'] - base_fail)
                   .query('count >= 30')
                   .sort_values(['fail_uplift', 'count'], ascending=[False, False]))

print('Q10 - Strongest failure-indicator codes (uplift over baseline)')
display(code_fail_stats.head(10))

## Modeling (F1-Oriented)

In [ ]:
# Build sparse violation dictionaries from frequent codes
top_codes = train_df['viol_codes'].explode().dropna().value_counts().head(80).index.tolist()

def to_violation_dict(series, keep_codes):
    keep = set(keep_codes)
    out = []
    for codes in series:
        d = {}
        for c in codes:
            if c in keep:
                d[f'viol_{c}'] = 1
        out.append(d)
    return out

train_df['viol_dict'] = to_violation_dict(train_df['viol_codes'], top_codes)
test_df['viol_dict'] = to_violation_dict(test_df['viol_codes'], top_codes)

feature_cols = [
    'Facility Type', 'Risk', 'City', 'State', 'Inspection Type',
    'Zip', 'Latitude', 'Longitude', 'year', 'month', 'weekday',
    'viol_count', 'unique_viol_count', 'has_any_violation', 'viol_dict'
]

X = train_df[feature_cols].copy()
y = train_df['target'].astype(int)

cat_cols = ['Facility Type', 'Risk', 'City', 'State', 'Inspection Type']
num_cols = ['Zip', 'Latitude', 'Longitude', 'year', 'month', 'weekday', 'viol_count', 'unique_viol_count', 'has_any_violation']

def cv_f1_score(X_all, y_all):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []

    for tr_idx, va_idx in skf.split(X_all, y_all):
        X_tr = X_all.iloc[tr_idx].copy()
        X_va = X_all.iloc[va_idx].copy()
        y_tr = y_all.iloc[tr_idx]
        y_va = y_all.iloc[va_idx]

        d_tr = X_tr.pop('viol_dict')
        d_va = X_va.pop('viol_dict')

        vec = DictVectorizer(sparse=True)
        Xv_tr = vec.fit_transform(d_tr)
        Xv_va = vec.transform(d_va)

        pre = ColumnTransformer(
            transformers=[
                ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True), cat_cols),
                ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler(with_mean=False))]), num_cols),
            ],
            sparse_threshold=1.0
        )

        Xp_tr = pre.fit_transform(X_tr)
        Xp_va = pre.transform(X_va)

        from scipy.sparse import hstack
        Xtr_all = hstack([Xp_tr, Xv_tr]).tocsr()
        Xva_all = hstack([Xp_va, Xv_va]).tocsr()

        model = LogisticRegression(max_iter=2000, solver='liblinear', class_weight='balanced', random_state=42)
        model.fit(Xtr_all, y_tr)
        pred = model.predict(Xva_all)
        scores.append(f1_score(y_va, pred))

    return np.mean(scores), np.std(scores)

cv_mean, cv_std = cv_f1_score(X, y)
print(f'5-fold CV F1 mean: {cv_mean:.5f}')
print(f'5-fold CV F1 std: {cv_std:.5f}')

In [ ]:
# Final training on full data and submission export
X_train = train_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()

d_train = X_train.pop('viol_dict')
d_test = X_test.pop('viol_dict')

vec = DictVectorizer(sparse=True)
Xv_train = vec.fit_transform(d_train)
Xv_test = vec.transform(d_test)

pre = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True), cat_cols),
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler(with_mean=False))]), num_cols),
    ],
    sparse_threshold=1.0
)

Xp_train = pre.fit_transform(X_train)
Xp_test = pre.transform(X_test)

from scipy.sparse import hstack
Xall_train = hstack([Xp_train, Xv_train]).tocsr()
Xall_test = hstack([Xp_test, Xv_test]).tocsr()

final_model = LogisticRegression(max_iter=2000, solver='liblinear', class_weight='balanced', random_state=42)
final_model.fit(Xall_train, y)
test_pred = final_model.predict(Xall_test).astype(int)

submission = pd.DataFrame({'id': test_df['id'].astype(int), 'target': test_pred})
submission = sample[['id']].merge(submission, on='id', how='left')
submission.to_csv('submission.csv', index=False)

print('submission.csv created.')
submission.head(10)

## Conclusion

- The model uses categorical features, numeric features, and engineered violation features.
- Class imbalance is handled with `class_weight='balanced'`.
- The notebook answers all 10 required analytical prompts and generates a valid submission file.

You can improve further with model ensembling (e.g., gradient boosting + linear model), rare-category grouping, and threshold tuning for maximizing F1.